# Lab Session 04 - 23CSE301
A1: Lab 03 experiments regenerated with AI-tool assistance (Claude)
A2: Unit tests for Lab 03 and Lab 04 functions
A3: Performance comparison - own k-means vs AI-generated k-means

NOTE: Function signatures / modularization are my own design choices (as required).
AI (Claude) was used to help generate/optimize the internal implementation.
Comments below mark which parts used AI assistance.

In [1]:
import time
import unittest
import numpy as np
import pandas as pd

## A1. AI-assisted re-implementation of Lab 03 functions
(Generated with help of Claude, vectorized versions of my own Lab03 functions)

In [2]:
def label_encode_ai(column):
    """AI-assisted: same behaviour as Lab03 label_encode, vectorized with pandas.factorize."""
    codes, uniques = pd.factorize(column, sort=True)
    mapping = {cat: idx for idx, cat in enumerate(uniques)}
    return codes, mapping


def one_hot_encode_ai(column):
    """AI-assisted: uses pandas.get_dummies instead of manual loop for speed."""
    return pd.get_dummies(column, prefix=column.name).astype(int)


def minkowski_distance_ai(vec1, vec2, p):
    """AI-assisted vectorized Minkowski distance (same formula as Lab03, numpy-only)."""
    vec1 = np.asarray(vec1, dtype=float)
    vec2 = np.asarray(vec2, dtype=float)
    return np.power(np.sum(np.power(np.abs(vec1 - vec2), p)), 1.0 / p)


def dot_product_ai(vec1, vec2):
    return float(np.dot(vec1, vec2))


def euclidean_norm_ai(vec):
    return float(np.linalg.norm(vec))


def dataset_stats_ai(matrix):
    """AI-assisted: numpy vectorized mean/std instead of the python-loop version in Lab03."""
    return matrix.mean(axis=0), matrix.std(axis=0)


# ---- AI-assisted vectorized K-Means (Claude-generated, optimized with broadcasting) ----
def kmeans_ai(data, k, max_iters=100, tol=1e-6, seed=42):
    rng = np.random.default_rng(seed)
    centroids = data[rng.choice(data.shape[0], size=k, replace=False)].copy()

    for iteration in range(max_iters):
        # broadcasting distance calc instead of nested python loops (AI suggestion)
        dists = np.linalg.norm(data[:, None, :] - centroids[None, :, :], axis=2)
        labels = np.argmin(dists, axis=1)

        new_centroids = np.array([
            data[labels == c].mean(axis=0) if np.any(labels == c) else centroids[c]
            for c in range(k)
        ])

        if np.sum(np.abs(new_centroids - centroids)) < tol:
            centroids = new_centroids
            break
        centroids = new_centroids

    return labels, centroids, iteration + 1

## A2. Unit tests for both Lab03 (own) and Lab04 (AI-assisted) functions

In [3]:
class TestDistanceFunctions(unittest.TestCase):
    def setUp(self):
        self.vec1 = np.array([1.0, 2.0, 3.0])
        self.vec2 = np.array([4.0, 6.0, 8.0])

    def test_minkowski_manhattan(self):
        expected = 3 + 4 + 5  # |1-4|+|2-6|+|3-8|
        self.assertAlmostEqual(minkowski_distance_ai(self.vec1, self.vec2, p=1), expected)

    def test_minkowski_euclidean(self):
        expected = (3**2 + 4**2 + 5**2) ** 0.5
        self.assertAlmostEqual(minkowski_distance_ai(self.vec1, self.vec2, p=2), expected)

    def test_dot_product(self):
        expected = 1*4 + 2*6 + 3*8
        self.assertAlmostEqual(dot_product_ai(self.vec1, self.vec2), expected)

    def test_euclidean_norm(self):
        expected = (1**2 + 2**2 + 3**2) ** 0.5
        self.assertAlmostEqual(euclidean_norm_ai(self.vec1), expected)


class TestEncodingFunctions(unittest.TestCase):
    def setUp(self):
        self.col = pd.Series(['A', 'B', 'A', 'C'], name='cat')

    def test_label_encode_unique_codes(self):
        codes, mapping = label_encode_ai(self.col)
        self.assertEqual(len(set(codes)), 3)
        self.assertEqual(mapping['A'], 0)

    def test_one_hot_shape(self):
        ohe = one_hot_encode_ai(self.col)
        self.assertEqual(ohe.shape, (4, 3))
        self.assertEqual(ohe.sum().sum(), 4)  # one hot per row


class TestStatsFunctions(unittest.TestCase):
    def test_dataset_stats(self):
        matrix = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)
        means, stds = dataset_stats_ai(matrix)
        np.testing.assert_almost_equal(means, [3.0, 4.0])
        np.testing.assert_almost_equal(stds, matrix.std(axis=0))


class TestKMeans(unittest.TestCase):
    def test_kmeans_runs_and_returns_valid_labels(self):
        data = np.vstack([
            np.random.default_rng(0).normal(0, 0.5, size=(20, 2)),
            np.random.default_rng(1).normal(10, 0.5, size=(20, 2)),
        ])
        labels, centroids, n_iters = kmeans_ai(data, k=2)
        self.assertEqual(len(labels), 40)
        self.assertEqual(centroids.shape, (2, 2))
        self.assertTrue(n_iters <= 100)

## A3. Performance comparison: own k-means (Lab03) vs AI-assisted k-means (Lab04)

In [4]:
def own_kmeans_for_timing(data, k, max_iters=100, tol=1e-6, seed=42):
    """Copy of the Lab03 loop-based k-means, kept here so this file is self-contained for timing."""
    rng = np.random.default_rng(seed)
    centroids = data[rng.choice(data.shape[0], size=k, replace=False)].copy()

    def dist(a, b):
        return np.sum(np.abs(a - b) ** 2) ** 0.5

    for iteration in range(max_iters):
        labels = np.zeros(data.shape[0], dtype=int)
        for i, point in enumerate(data):
            d = [dist(point, c) for c in centroids]
            labels[i] = int(np.argmin(d))

        new_centroids = np.zeros((k, data.shape[1]))
        for c in range(k):
            members = data[labels == c]
            new_centroids[c] = members.mean(axis=0) if len(members) else centroids[c]

        if np.sum(np.abs(new_centroids - centroids)) < tol:
            centroids = new_centroids
            break
        centroids = new_centroids

    return labels, centroids, iteration + 1


def compare_kmeans_performance(data, k, n_runs=3):
    own_times, ai_times = [], []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        own_kmeans_for_timing(data, k)
        own_times.append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        kmeans_ai(data, k)
        ai_times.append(time.perf_counter() - t0)

    return {
        "own_mean_time": np.mean(own_times),
        "ai_mean_time": np.mean(ai_times),
        "speedup": np.mean(own_times) / np.mean(ai_times),
    }

## Main program

In [5]:
if __name__ == "__main__":
    # run unit tests
    suite = unittest.TestLoader().loadTestsFromModule(__import__('__main__'))
    runner = unittest.TextTestRunner(verbosity=2)
    runner.run(suite)

    # performance comparison on marketing_campaign numeric data
    raw_df = pd.read_excel("Lab_Session_Data.xlsx", sheet_name="marketing_campaign")
    numeric_df = raw_df.select_dtypes(include=[np.number]).drop(columns=['ID']).dropna()
    data = numeric_df.to_numpy(dtype=float)
    col_std = data.std(axis=0)
    col_std[col_std == 0] = 1  # guard against constant columns like Z_CostContact/Z_Revenue
    data = (data - data.mean(axis=0)) / col_std  # standardize

    results = compare_kmeans_performance(data, k=4, n_runs=3)
    print("\nPerformance comparison (own vs AI-assisted k-means):")
    print(f"  Own (loop-based)      avg time: {results['own_mean_time']:.4f} s")
    print(f"  AI-assisted (vectorized) avg time: {results['ai_mean_time']:.4f} s")
    print(f"  Speedup (own/ai): {results['speedup']:.2f}x")

test_dot_product (__main__.TestDistanceFunctions.test_dot_product) ... 

ok


test_euclidean_norm (__main__.TestDistanceFunctions.test_euclidean_norm) ... 

ok


test_minkowski_euclidean (__main__.TestDistanceFunctions.test_minkowski_euclidean) ... 

...

ok


test_dataset_stats (__main__.TestStatsFunctions.test_dataset_stats) ... 

ok


----------------------------------------------------------------------
Ran 8 tests in 0.031s

OK



Performance comparison (own vs AI-assisted k-means):
  Own (loop-based)      avg time: 1.3804 s
  AI-assisted (vectorized) avg time: 0.0673 s
  Speedup (own/ai): 20.51x
